# Solar Installation Timeline in California

This notebook calculates the median time from solar interconnection application to utility approval (Permission to Operate) using public CPUC data downloadable from [DGStats](https://www.californiadgstats.ca.gov/downloads/). At the time of download, the latest accessible data from DGStats was from May 31, 2026.

This analysis is done by [SolarWAVE Action](https://solarwaveaction.org/), a California based nonprofit protecting clean energy ownership. All our code is open source and can be found in our [GitHub](https://github.com/SolarWAVE-Action/solarwave-analysis) repository.
For any questions or comments, please reach out to hello@solarwaveaction.org.

Below, Python packages are imported and the application data from Pacific Gas & Electric (PG&E), Southern California Edison (SCE) and San Diego Gas & Electric (SDG&E) are consolidated. 

In [1]:
import datetime
from IPython.display import HTML
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

import data_analysis as data_analysis
import data_visualization as data_vis

pio.templates.default = 'plotly_white'
pio.renderers.default = 'notebook'

In [2]:
data_dir = '/Users/jenny.folkesson/Data/solar/'
dgstats_dir = os.path.join(data_dir, 'applications_20260531/')
logo_path = "https://solarwaveaction.org/assets/images/SolarWAVEActionLogoTransparent.png"

df_total = data_analysis.read_stored_data(dgstats_dir, "dgstats_preprocessed_data.csv")

Reading existing file:  /Users/jenny.folkesson/Data/solar/applications_20260531/dgstats_preprocessed_data.csv


DGStats have 3 time related metrics: 
- 'App Recieved Date': The date the application was received by the utility
- 'App Complete Date': The date the application was deemed complete by the utility
- 'App Approved Date': The date the application was approved and a Permission to Operate (PTO) was issued to the customer

The clearest picture of how long a solar application takes in California is obtained by finding the number of days between App Recieved Date and App Approved Date.

In [3]:
df_total['App Days'] = ((df_total['App Approved Date'] - df_total['App Received Date']).dt.days)

Next, the median number of application days ('App Days') are calculated on an annual basis.

In [4]:
df = df_total[['App Approved Date', 'System Size DC', 'App Days']]
df = df.set_index('App Approved Date').rename_axis(None)
df = df.resample("YE").agg({'System Size DC': 'sum', 'App Days': 'median'})
df['Year'] = df.index
df['Year'] = df['Year'].dt.year
df = df[df['Year'] > 2016]

Below, the median solar installation times in days for all customer sectors (including residential, commercial, industrial, non-profit, educational, military and other government) is displayed for each year for the last five years (2026 is incomplete).

In [5]:
df[['Year', 'App Days']]

,Year,App Days
2017-12-31,2017,5.0
2018-12-31,2018,4.0
2019-12-31,2019,6.0
2020-12-31,2020,7.0
2021-12-31,2021,11.0
2022-12-31,2022,11.0
2023-12-31,2023,101.0
2024-12-31,2024,30.0
2025-12-31,2025,26.0
2026-12-31,2026,29.0


The median time for a solar installation from the day the application was received to complete is 29 days for projects completed in 2026 so far across all customer sectors.

This can be compared to numbers from the National Laboratory of the Rockies (NLR), which was formerly known as National Renewable Energy Laboratory (NREL) but was renamed by the Trump administation. Their [SolarTRACE Data Viewer](https://maps.nlr.gov/solarTRACE/data-viewer) shows a median timeline for all California solar project sizes in **2025 of 26 days** (found by adding the median numbers for CA Permitting, CA Pre-Interconnection, CA - Inspection, and CA - Post Interconnection).
That is the exact same median time that this analysis found for 2025; 26 days.

The increase in installation times in 2023 can be attributable to the implementation of the [Net Billing Tariff](https://www.cpuc.ca.gov/industries-and-topics/electrical-energy/demand-side-management/customer-generation/net-energy-metering-and-net-billing), which caused a spike in solar applications that year. 

**SolarWAVE Action found a median installation time of 29 days in 2026 for California solar projects across all customer sectors.**
Our analysis method is shared openly for inspection and is consistent with findings by NLR. This median time is largely driven by the residential sector, the by far largest customer segment by application volume.

Below, the median application time is visualized in a bar graph for each customer sector.

In [6]:
fig = data_vis.application_time_linechart(df_total, date_type='App Approved Date')

html_str = fig.to_html(include_plotlyjs='cdn', full_html=False)
HTML(html_str)

The median application time for residential projects completed in 2026 so far is 27 days. Residential application times have stayed fairly flat over the years, as opposed to all other sectors, where application times have increased massively starting in 2022. Commercial applications completed in 2026 so far have typically spent 990 days in the application process. This wasn't always the case; commercial applications completed 5 years ago (in 2021) had a typical applicatin time of 157 days (5 months).

All but residental applications have experienced much longer application times since 2022. Let's look at application volume to see if the utilities are perhaps processing more applications now, which could lead to projects stuck long in the queue showing up now.

In [7]:
fig = data_vis.application_time_linechart(df_total, date_type='App Approved Date', y_label='Count', hidden_sectors=['Residential'])

html_str = fig.to_html(include_plotlyjs='cdn', full_html=False)
HTML(html_str)

Above, the line graph shows the number of applications completed per year per customer sector. Residential applications are hidden because there are 100x residential applications compared to other customer sectors, but you can see residential applications as well by clicking on the legend on the right hand side. 

What we can tell from the two graphs above is that aside from residential and commercial applications, application volumes are fairly constant. 

While commercial solar completions dropped by 16% from 2024-2025, the median application time still increased from 693 to 990 days. This raises the question: why are all solar applications aside from residential taking so long to complete?

We can pack more information into one graph. In the graph below, the y-axis still represents application time, but the lines are replaced by circles, where the circle area represents the number of applications.

In [9]:
fig = data_vis.application_time_bubble(df_total, years=10)

# Write figure
title_text = "Non-residential solar application times have risen several-fold since 2022"
caption_text = "Median days from interconnection application to permission to operate, by customer sector and year of approval,<br> \
for PG&E, SCE, and SDG&E (2026 data through May 31). Residential solar (circle area is proportional to application volume) has<br> \
stayed near or under a month except during the 2023 application rush ahead of the Net Billing Tariff, while non-residential<br> \
sectors have seen median times rise several-fold since 2022. <br>Source: SolarWAVE Action analysis of CPUC DGStats data."
write_path = os.path.join(data_dir, "customer_sector_application_time")
data_vis.write_fig(fig, write_path, title=title_text, caption=caption_text, logo_path=logo_path, logo_x=1.2)

html_str = fig.to_html(include_plotlyjs='cdn', full_html=False)
HTML(html_str)

This graph shows that the residential solar processing times increased slightly in 2023 with the NEM 2.0 cutoff, following an increase in application volume leading up to that event. Residential application times have since dropped again.

This is not the case for any of the other customer categories, such as commercial and industrial. Despite fairly constant or even decreasing application volume, application times have steadily increased since 2022. A typical industrial project approved today has spent almost 3 years in the application process. Fewer commercial applications were completed in 2025 than the year before, yet the median time to complete increased.

A big problem is that DGStats only publishes completed applications, which means we see a delayed snapshot of what's happening with California's distributed solar. This delay can be several years for non-residential customer sectors. Publishing pending applications would offer a much better overview of the current state of the distributed solar market.